# Séance 1 – Exercices
**Ingénierie logicielle : POO avancée, SOLID, Clean Code**

Enseignant : Jean Delpech

Cours : Data Science

Classe : M1 Data/IA et Data/E

Année scolaire : 2025/2026

Dernière mise à jour : mai 2026

---

## Liste des exercices

1. [Diagnostic : identifier les violations dans du code data](#Exercice-1)
2. [SRP : Refactoring d'une classe God Object](#Exercice-2)
3. [OCP : Extensibilité sans modification](#Exercice-3)
4. [LSP : Contrats et substitution](#Exercice-4)
5. [DIP : Injection de dépendance](#Exercice-5)
6. [Composition vs héritage](#Exercice-6)
7. [Clean Code : Nommage, type hints, docstrings](#Exercice-7)
8. [Mini-projet](#Exercice-8)

## Exercice 1
**Diagnostic : identifier les violations**

Le code ci-dessous est représentatif de ce qu'on trouve fréquemment dans des projets data.
Lisez-le attentivement, puis répondez aux questions.

In [ ]:
# Code à analyser - ne pas modifier cette cellule

import pandas as pd
import smtplib
import json
import psycopg2

class DataManager:
    """
    Gère tout ce qui touche aux données.
    """

    def __init__(self):
        self.conn = psycopg2.connect(
            host='localhost', dbname='prod', user='admin', password='s3cr3t'
        )
        self.smtp = smtplib.SMTP('smtp.company.com', 587)
        self.data = None
        self.model = None

    def process(self, path, format, send_email=False):
        # Chargement
        if format == 'csv':
            self.data = pd.read_csv(path)
        elif format == 'json':
            self.data = pd.read_json(path)
        elif format == 'parquet':
            self.data = pd.read_parquet(path)
        else:
            raise ValueError(f"Format inconnu : {format}")

        # Nettoyage
        self.data = self.data.dropna()
        self.data = self.data.drop_duplicates()
        for col in self.data.select_dtypes(include='object').columns:
            self.data[col] = self.data[col].str.strip().str.lower()

        # Entraînement d'un modèle basique
        from sklearn.linear_model import LogisticRegression
        X = self.data.drop('target', axis=1)
        y = self.data['target']
        self.model = LogisticRegression()
        self.model.fit(X, y)
        score = self.model.score(X, y)

        # Sauvegarde en base
        cur = self.conn.cursor()
        cur.execute("INSERT INTO results (score) VALUES (%s)", (score,))
        self.conn.commit()

        # Envoi d'email
        if send_email:
            msg = f"Score du modèle : {score:.4f}"
            self.smtp.sendmail('pipeline@company.com', 'team@company.com', msg)

        return score

    def predict(self, new_data):
        if self.model is None:
            raise RuntimeError("Modèle non entraîné")
        if isinstance(new_data, dict):
            new_data = pd.DataFrame([new_data])
        elif isinstance(new_data, list):
            new_data = pd.DataFrame(new_data)
        return self.model.predict(new_data)

print("Code chargé. À vous d'analyser.")

## Questions de diagnostic

Répondez dans la cellule Markdown ci-dessous.

**1. SRP** – Listez toutes les responsabilités de la classe `DataManager`. Combien de raisons de changer peut-elle avoir ?

**2. OCP** – Quel bloc de code devra être modifié chaque fois qu'on ajoute un nouveau format de fichier ? Comment y remédier ?

**3. LSP** – La méthode `predict` contient des `isinstance`. Que signale ce pattern ? Quel principe est violé ?

**4. DIP** – Quelles dépendances concrètes sont câblées dans `__init__` ? Quels problèmes cela pose-t-il pour les tests ?

**5. Clean Code** – Identifiez au moins trois problèmes de lisibilité ou de nommage.

## Vos réponses

*Modifiez cette cellule pour répondre.*

**1. Responsabilités de DataManager :**
- ...

**2. Violation OCP :**
- ...

**3. Violation LSP :**
- ...

**4. Violation DIP :**
- ...

**5. Problèmes Clean Code :**
- ...

## Exercice 2
**SRP : Refactoring d'une classe God Object**

Découpez `DataManager` en classes à responsabilité unique.
Chaque classe doit avoir un nom clair et une seule raison de changer.

> Pour l'instant, ne vous préoccupez pas des implémentations complètes.
> Définissez la structure (classes, méthodes, signatures) et documentez avec des docstrings.

In [ ]:
from abc import ABC, abstractmethod
import pandas as pd
from dataclasses import dataclass

# TODO : définir les classes ci-dessous
# Chaque classe doit avoir :
#   - Un nom révélateur de son rôle
#   - Une docstring décrivant sa responsabilité unique
#   - Des méthodes avec type hints et docstrings

# Classe 1 : chargement des données
class DataLoader:
    """
    TODO : décrire la responsabilité
    """
    def load(self, path: str) -> pd.DataFrame:
        ...

# Classe 2 : nettoyage
# TODO

# Classe 3 : entraînement du modèle
# TODO

# Classe 4 : persistance
# TODO

# Classe 5 : notification
# TODO

print("Structure à compléter.")

## Exercice 3
**OCP : Extensibilité sans modification**

La fonction `export()` ci-dessous viole l'OCP : ajouter un format impose de la modifier.
Refactorisez en utilisant une abstraction, de sorte que l'ajout du format Parquet
ne nécessite **aucune modification** du code existant.

In [ ]:
import pandas as pd
import json
import io

# Version initiale – viole OCP
def export_v1(df: pd.DataFrame, format: str, path: str) -> None:
    if format == 'csv':
        df.to_csv(path, index=False)
    elif format == 'json':
        df.to_json(path, orient='records')
    # Ajouter Parquet ici oblige à modifier cette fonction


# TODO : refactoriser avec une abstraction Exporter
# Contrainte : ajouter ParquetExporter ne doit pas toucher au code existant

from abc import ABC, abstractmethod

class Exporter(ABC):
    @abstractmethod
    def export(self, df: pd.DataFrame, path: str) -> None:
        ...

# TODO : implémenter CSVExporter, JSONExporter, puis ParquetExporter

print("À compléter.")

## Exercice 4
**LSP : Contrats et substitution**

L'exemple classique `Rectangle` / `Square` illustre une violation du LSP.
Après l'avoir analysée, proposez une alternative qui respecte le principe.

In [ ]:
# Violation LSP - à analyser - ne pas modifier

class Rectangle:
    def __init__(self, width: float, height: float):
        self._width  = width
        self._height = height

    def set_width(self, w: float) -> None:
        self._width = w

    def set_height(self, h: float) -> None:
        self._height = h

    def area(self) -> float:
        return self._width * self._height


class Square(Rectangle):
    """Un carré est un rectangle avec width == height."""

    def set_width(self, w: float) -> None:
        self._width  = w
        self._height = w   # effet de bord inattendu

    def set_height(self, h: float) -> None:
        self._width  = h   # effet de bord inattendu
        self._height = h


def assert_rectangle_behavior(shape: Rectangle) -> None:
    """
    Postcondition attendue pour un Rectangle :
    après set_width(5) et set_height(3), l'aire doit être 15.
    """
    shape.set_width(5)
    shape.set_height(3)
    area = shape.area()
    assert area == 15, f"Violation du contrat Rectangle : aire={area}, attendu=15"
    print(f"Contrat respecté : aire = {area}")


# Test avec Rectangle : OK
assert_rectangle_behavior(Rectangle(1, 1))

# Test avec Square : viole le contrat
try:
    assert_rectangle_behavior(Square(1, 1))
except AssertionError as e:
    print(f"Violation LSP détectée : {e}")

In [ ]:
# Solution : utiliser la composition ou une hiérarchie sans mutation
# TODO : proposer une alternative qui respecte le LSP

# Piste 1 : rendre Rectangle immuable (pas de setters)
# Piste 2 : Shape abstraite avec area() – Rectangle et Square en sont des implémentations indépendantes
# Piste 3 : Square ne dérive pas de Rectangle



## Exercice 5
**DIP : Injection de dépendance**

Cet exercice est théoriquement très complexe, DIP est le principe le plus abstrait, de plus cet exercice aborde la théorie des tests, pas de panique si vous n’y arrivez pas. J’essaye de vous guider au maximum.

Voici un pipeline de nettoyage de données tel qu'on en écrit souvent quand on démarre un projet :

In [ ]:
import pandas as pd
import psycopg2

class CleaningPipeline:
    """Pipeline de nettoyage – version initiale, non testable."""

    def __init__(self):
        # Dépendances concrètes câblées en dur dans le constructeur
        self._conn = psycopg2.connect(
            host="prod-db.company.com",
            dbname="datawarehouse",
            user="etl_user",
            password="s3cr3t",
        )
        self._output_path = "/data/exports/cleaned.csv"

    def run(self) -> None:
        """Lit depuis la BDD, nettoie, écrit sur disque, enregistre une métrique."""
        # Lecture
        df = pd.read_sql("SELECT * FROM raw_customers", self._conn)
        n_before = len(df)

        # Nettoyage
        df = df.dropna().drop_duplicates()

        # Écriture sur le disque local
        df.to_csv(self._output_path, index=False)

        # Métrique brute dans la console
        print(f"Lignes avant : {n_before} – après : {len(df)}")

Ce code :
- lit une table SQL,
- nettoie les données (suppression des nulls et des doublons)
- et écrit le résultat dans un fichier CSV local, en affichant le nombre de lignes avant et après.

Exécuter ce pipeline en test nécessite une vraie base PostgreSQL de production accessible et un chemin disque valide. C'est impossible en CI/CD et dangereux en cours de développement.

Pour résoudre ce problème, on va créer des interfaces.

Il faut donc :

1. Identifier les trois dépendances concrètes câblées dans cette classe (source, destination, collecteur de métriques) et définir pour chacune une abstraction (ABC) avec une interface minimale.
2. Réécrire `CleaningPipeline` pour qu'elle reçoive ces abstractions par injection dans son __init__, sans plus aucune référence à `psycopg2`, au disque local ou à `print`.
3. Implémenter pour chaque abstraction un test (par exemple `InMemorySource`...) qui fonctionne entièrement en mémoire.
4. Démontrer que le pipeline fonctionne sans aucune infrastructure externe en l'instanciant avec les doubles de test et en vérifiant que les données produites sont correctes.

> Implémenter un test ?
>
> Au point 3 on demander l’implémentation simplifiée d'une abstraction, conçue uniquement pour les tests : elle respecte le contrat de l'interface (les mêmes signatures de méthodes) mais remplace la vraie infrastructure par du contenu en mémoire vive. C'est l'équivalent d'un "stub" (bouchon) qu'on branche à la place du vrai composant le temps de vérifier que le pipeline fait ce qu'il doit faire.
> Dans cet exercice, vous allez en écrire trois :
> - `InMemorySource` reçoit un DataFrame au moment de sa construction et le retourne quand on appelle ̀ read()`, elle simule la base PostgreSQL.
> - `InMemorySink` ne fait pas qu'accepter les données : elle les garde dans une liste `written` que vous pouvez inspecter après l'exécution pour vérifier que le pipeline a bien produit le bon résultat, elle simule le fichier CSV.
> - `InMemoryMetrics` accumule les métriques dans un dictionnaire records, elle simule ce qui serait en production un appel à `Datadog`, `Prometheus` ou un simple log fichier.
> 
> L'idée centrale ici est que ce qui entre dans chacun de ces « double de test » (vu qu’ils remplacent quelque chose pour le tester) est fixé à la construction (__init__), ce qui sort est accessible après l'exécution via un attribut public. C'est ce qui rend le test possible sans infrastructure : on prépare les données d'entrée, on lance le pipeline, on lit les sorties dans les attributs.
>
> Pour une présentation plus formelle et systématique de cette méthodologie je vous renvoie vers Gerard Meszaros http://xunitpatterns.com/gerardmeszaros.html (catégorie « Test double »). Vous rencontrerez souvent le terme de `mock` car c’est celui le plus souvent repris dans des bibliothèques de test ([unittest.mock](https://docs.python.org/3/library/unittest.mock.html) en Python, [Mockery](https://pieces-of-code.com/guide/quickstart/mockito.html#les-concepts-de-base) en Java…), même si les puristes feront la distinction entre mock, stub, fake, dummy…

Contraintes :

- `CleaningPipeline` ne doit contenir aucun `import psycopg2`, aucun chemin disque, aucun `print`
- Les trois abstractions doivent être des `ABC` avec `@abstractmethod`
- Les doubles de test doivent permettre d'inspecter ce qui a été écrit et quelles métriques ont été enregistrées
- Type hints et docstrings sur toutes les classes et méthodes publiques

In [ ]:
import pandas as pd
from abc import ABC, abstractmethod


# ── ÉTAPE 1 : Définir les trois abstractions ──────────────────────
# Chaque abstraction = une interface minimale que le pipeline utilise
# sans connaître l'implémentation concrète derrière.

class DataSource(ABC):
    """Abstraction pour toute source de données tabulaires."""

    @abstractmethod
    def read(self) -> pd.DataFrame:
        """Lit les données et les retourne sous forme de DataFrame."""
        ...


class DataSink(ABC):
    """Abstraction pour toute destination des données nettoyées."""

    @abstractmethod
        # TODO : définir la signature et la docstring pour write()
        # rappel POO : la signature d’une méthode est la première ligne d’une méthode
        # c’est-à-dire la ligne 'def' avec les paramètres (et leurs types) et le type que
        # la méthode doit retourner ('-> type_retour')
        ...


class MetricsCollector(ABC):
    """Abstraction pour l'enregistrement des métriques du pipeline."""

    @abstractmethod
        # TODO : définir la signature et la docstring pour record()
        ...


# ── ÉTAPE 2 : Implémenter les trois versions de test (InMemory) ───
# Ces classes remplacent la vraie infrastructure pendant les tests.
# Ce qui entre est fixé à la construction (__init__).
# Ce qui sort est accessible après l'exécution via un attribut public.

class InMemorySource(DataSource):
    """Simule une source de données – remplace PostgreSQL en test."""

    def __init__(self, df: pd.DataFrame):
        self._df = df

    def read(self) -> pd.DataFrame:
        # TODO : retourner une copie du DataFrame stocké
        ...


class InMemorySink(DataSink):
    """Simule une destination – remplace le fichier CSV en test.
    
    Après l'exécution du pipeline, inspecter self.written
    pour vérifier que les données produites sont correctes.
    """

    def __init__(self):
        self.written: list[pd.DataFrame] = []  # attribut d'inspection

    def write(self, df: pd.DataFrame) -> None:
        # TODO : stocker df dans self.written
        ...


class InMemoryMetrics(MetricsCollector):
    """Simule un collecteur de métriques – remplace Datadog/logs en test.
    
    Après l'exécution du pipeline, inspecter self.records
    pour vérifier que les bonnes métriques ont été enregistrées.
    """

    def __init__(self):
        self.records: dict[str, list[float]] = {}  # attribut d'inspection

    def record(self, metric_name: str, value: float) -> None:
        # TODO : accumuler value dans self.records[metric_name]
        ...


# ── ÉTAPE 3 : Réécrire le pipeline avec injection de dépendance ───
# CleaningPipeline ne doit contenir aucune référence à psycopg2,
# au disque local, ou à print(). Il reçoit tout par son __init__.

class CleaningPipeline:
    """
    Pipeline de nettoyage de données.
    Toutes les dépendances sont injectées – aucune infrastructure câblée.
    """

    def __init__(
        self,
        source:  DataSource,       # TODO : compléter les deux autres
        # ...
    ):
        self._source = source
        # TODO : stocker les autres dépendances

    def run(self) -> None:
        """Lit, nettoie et persiste les données en enregistrant les métriques."""
        df = self._source.read()

        # TODO :
        # 1. Enregistrer le nombre de lignes avant nettoyage
        # 2. Nettoyer (dropna + drop_duplicates)
        # 3. Enregistrer le nombre de lignes après nettoyage
        # 4. Écrire le résultat via self._sink
        ...


# ── ÉTAPE 4 : Démontrer que ça fonctionne sans infrastructure  externe -----------

sample_data = pd.DataFrame({
    "name":  ["Alice", "Bob", None, "Alice"],
    "score": [0.92, 0.87, 0.75, 0.92],
})

#  instancier les trois InMemory..., créer le pipeline, appeler run()

source  = InMemorySource(sample_data)
sink    = InMemorySink()
metrics = InMemoryMetrics()

pipeline = CleaningPipeline(source, sink, metrics)
pipeline.run()

# 1. afficher sink.written[0] et metrics.records
# 2. écrire 3 assertions qui vérifient que le résultat est correct
#   - le nombre de lignes après nettoyage
#   - l'absence de valeurs nulles
#   - l'absence de doublons

# Vérification des résultats
print("Données nettoyées :")
print(sink.written[0])

print("\nMétriques enregistrées :")
for name, values in metrics.records.items():
    print(f"  {name} : {values[0]:.0f}")

# Assertions – le pipeline produit bien le résultat attendu
result = sink.written[0]
assert len(result) == 2,          "Attendu 2 lignes après suppression null + doublon"
assert result["name"].notna().all(), "Aucune valeur nulle attendue"
assert not result.duplicated().any(), "Aucun doublon attendu"

print("\nTous les tests passent – aucune connexion PostgreSQL, aucun fichier disque.")

## Exercice 6
**Composition vs héritage**

On veut modéliser différentes stratégies de nettoyage de données.
Comparez les deux approches et identifiez pourquoi la composition est préférable ici.

In [ ]:
import pandas as pd
from abc import ABC, abstractmethod

# --- Approche 1 : héritage -----------
class BaseCleaner:
    def clean(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.dropna()

class DeduplicatingCleaner(BaseCleaner):
    def clean(self, df: pd.DataFrame) -> pd.DataFrame:
        df = super().clean(df)
        return df.drop_duplicates()

class NormalizingDeduplicatingCleaner(DeduplicatingCleaner):
    def clean(self, df: pd.DataFrame) -> pd.DataFrame:
        df = super().clean(df)
        for col in df.select_dtypes(include='object').columns:
            df[col] = df[col].str.lower().str.strip()
        return df

# Problème : et si on veut normalisation SANS dédupliquer ?
# Il faut créer une autre branche dans la hiérarchie.
# La hiérarchie explose combinatoirement.

print("Approche héritage : hiérarchie rigide.")


# --- Approche 2 : composition (pattern Strategy) -----------

class CleaningStep(ABC):
    """Une étape de nettoyage. Peut être composée avec d'autres."""
    @abstractmethod
    def apply(self, df: pd.DataFrame) -> pd.DataFrame: ...


class DropNullsStep(CleaningStep):
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.dropna()


class DropDuplicatesStep(CleaningStep):
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.drop_duplicates()


class NormalizeStringsStep(CleaningStep):
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        for col in df.select_dtypes(include='object').columns:
            df[col] = df[col].str.lower().str.strip()
        return df


class DataCleaner:
    """
    Pipeline de nettoyage composable : on choisit les étapes à l'instanciation.
    Chaque combinaison est possible sans créer de nouvelle sous-classe.
    """
    def __init__(self, steps: list[CleaningStep]):
        self._steps = steps

    def clean(self, df: pd.DataFrame) -> pd.DataFrame:
        for step in self._steps:
            df = step.apply(df)
        return df


# Démonstration : différentes combinaisons sans nouvelle classe
df = pd.DataFrame({
    'name':  ['Alice', 'BOB', None, 'Alice', ' Carol '],
    'score': [0.92, 0.87, 0.75, 0.92, 0.80],
})

configs = {
    "Nulls seulement":           [DropNullsStep()],
    "Nulls + doublons":          [DropNullsStep(), DropDuplicatesStep()],
    "Tout (nulls+déduplique+norme)": [DropNullsStep(), DropDuplicatesStep(), NormalizeStringsStep()],
    "Norme seule":               [NormalizeStringsStep()],
}

for label, steps in configs.items():
    result = DataCleaner(steps).clean(df)
    print(f"\n{label} ({len(result)} lignes) :")
    print(result.to_string(index=False))

**Éditez cette cellule pour expliquer pourquoi la composition est préférable dans cette situation**

La composition est ici préférable parce que… *(donnez une argumentation précise en reprenant des éléments de cette situation, pas en formulant seulement une généralité)* 

> Si vous avez du mal à démarrer, montrez l’intérêt de la composition pour exécuter différentes opération selon différentes combinaisons (combien de classes cela demanderait de créer en comparaison ?)

## Exercice 7
**Clean Code : nommage, type hints, docstrings**

Le code ci-dessous est fonctionnel mais illisible.
Réécrivez-le en appliquant les pratiques Clean Code.

In [ ]:
# Code fonctionnel mais illisible – à refactoriser

import pandas as pd
import numpy as np

def proc(d, t=0.5, r=False):
    x = d.copy()
    x = x.dropna()
    c = []
    for i, row in x.iterrows():
        v = 0
        for col in x.columns:
            if col != 'label':
                v += row[col] ** 2
        v = v ** 0.5
        if v > t:
            c.append(i)
    if r:
        return x.drop(index=c)
    return x.loc[c]


# Exemple d'utilisation
df = pd.DataFrame({
    'f1': [0.1, 0.9, np.nan, 0.3, 0.8],
    'f2': [0.2, 0.7, 0.5,   0.1, 0.9],
    'label': [0, 1, 0, 0, 1]
})
result = proc(df, t=0.7, r=False)
print(result)

In [ ]:
# TODO : réécrire la fonction proc() en version Clean Code
# Critères :
#   - Nom de fonction révélateur
#   - Paramètres nommés explicitement
#   - Type hints complets
#   - Docstring (description, args, returns)
#   - Pas de variable à une lettre sauf conventions mathématiques
#   - Factoriser le calcul de norme en sous-fonction

def YOUR_FUNCTION_NAME(...):
    ...

print("À compléter.")

## Exercice 8
**Mini-projet**

### Contexte

Vous devez concevoir un **connecteur de données générique** utilisable dans plusieurs projets.
Il doit pouvoir lire des données depuis différentes sources (CSV local, API HTTP, base de données)
et les valider avant de les retourner.

### Contraintes

- Respecter les 5 principes SOLID
- Utiliser la composition plutôt que l'héritage pour les validations
- Type hints complets, docstrings sur toutes les classes et méthodes publiques
- Le code doit être testable sans connexion réseau ni base de données

### Structure suggérée

```
DataConnector
  ├── source:    DataSource          (abstraction – CSV, HTTP, DB...)
  ├── validator: DataValidator       (composition de règles de validation)
  └── method read() -> pd.DataFrame

ValidationRule (ABC)
  ├── NoNullsRule
  ├── MinRowsRule
  └── RequiredColumnsRule
```

In [ ]:
# Votre code ici





## Ce que vous devez absolument retenir et adopter comme réflexes

À chaque fois qu'on implémente une structure de données ou un algorithme :

1. Cette classe a-t-elle **une seule responsabilité** ?
2. Mes dépendances sont-elles **injectées** (passée en paramètre lors de l’instanciation = on lui passe quelque chose qu’elle ne veut pas connaître sinon qu’elle respecte le contrat attendu) plutôt que câblées (codées en dur, généralement dans le `__init__` avec des appels directs = elle appelle elle même quelque chose défini et connu en son sein) ?
3. Mon code est-il **extensible sans modification** (OCP) ?
4. Ai-je utilisé **type hints et docstrings** sur toutes les méthodes publiques ?
5. Mes fonctions sont-elles **courtes et nommées explicitement** ?